In [1]:
import litellm
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    litellm.openai_key = os.getenv("OPENAI_API_KEY")

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [7]:
# Student task: Implement exact_match and compute EM
# TODO: Complete the sections marked with **********

preds = ["Lima", "ayacucho", "Cusco", "Arequipa"]
labels = ["lima", "Ayacucho", "Cusco", "Trujillo"]


def normalize(s: str) -> str:
    """Normalize the string by lowercasing and stripping whitespace."""
    return s.lower().strip()


def exact_match(pred: str, label: str) -> int:
    # return 1 if normalized strings are identical, else 0
    return_value = int(normalize(pred) == normalize(label))

    return return_value


em_scores = [exact_match(p, l) for p, l in zip(preds, labels)]
em = sum(em_scores) / len(em_scores)
print("EM:", em)

assert em == 0.75, (
    f"EM should be 0.75, but got {em}. Please check your exact_match function."
)


EM: 0.75


In [8]:
# Student task: Compute ROUGE-L using LCS length
# Complete the sections with **********


# Define candidate and reference texts
pred = "The capital of Peru is Lima"
label = "Lima is the capital of Peru"


# Import the evaluate library
from evaluate import load

# Load the ROUGE metric
rouge = load('rouge')

# Compute ROUGE scores
results = rouge.compute(predictions=[pred], references=[label])


assert isinstance(results, dict), (
    f"Results should be a dictionary, but got {type(results)}. See the evaluate library documentation for ROUGE usage."
)
keys = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
for key in keys:
    assert key in results, (
        f"Missing key '{key}' in results. Expected keys: {keys}. See the evaluate library documentation for ROUGE usage."
    )

results


/Users/abhishekbarve/learning/ai-ml/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'rouge1': np.float64(1.0),
 'rouge2': np.float64(0.6),
 'rougeL': np.float64(0.6666666666666666),
 'rougeLsum': np.float64(0.6666666666666666)}

In [11]:
# Student task: Write a semantically different prediction sentence and compute embeddings
# Complete the sections with **********

labels = ["Cusco is in Peru", "Ayacucho is a region", "Trujillo beaches are marvelous"]
preds = [
    "Peru includes Cusco",
    "Ayacucho is a department",
    # Write a sentence that is very semantically different from the prediction
    "I am a DemiGod",
]

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')


# Get the embeddings for each sentence
pred_embeddings = model.encode(preds)
label_embeddings = model.encode(labels)


assert pred_embeddings.shape == (3, 384), (
    f"Expected shape (3, 384), got {pred_embeddings.shape}"
)
assert label_embeddings.shape == (3, 384), (
    f"Expected shape (3, 384), got {label_embeddings.shape}"
)

pred_embeddings, label_embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9119.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(array([[ 0.06145665, -0.06237293, -0.03735717, ..., -0.00960594,
          0.03519067, -0.01402613],
        [-0.01852836, -0.03180064, -0.07411849, ..., -0.00197018,
          0.01199766,  0.0113026 ],
        [ 0.02208523,  0.0336188 ,  0.10765463, ..., -0.01735573,
         -0.06902133, -0.0638718 ]], shape=(3, 384), dtype=float32),
 array([[ 0.06701855, -0.04063963, -0.06178867, ...,  0.01089181,
         -0.01366577, -0.02568763],
        [ 0.08464754,  0.00272663, -0.06455817, ...,  0.04696645,
         -0.06039641, -0.00335863],
        [ 0.03886197, -0.0283124 , -0.02234175, ...,  0.00904637,
         -0.0284746 , -0.00952086]], shape=(3, 384), dtype=float32))

In [13]:
# Calculate the cosine similarity for each pair of embeddings
# No changes needed in this cell, but if it fails, check the above cell
import numpy as np

cosine_similarity = [
    # Cosine similarity for two vectors a and b is defined as:
    # cos_sim(a, b) = (a . b) / (||a|| * ||b||)
    # where (a . b) is the dot product of a and b,
    # and ||a|| and ||b|| are the magnitudes (norms) of vectors a and b respectively.
    float(
        np.dot(pred_embeddings[i], label_embeddings[i])
        / np.linalg.norm(pred_embeddings[i])
        / np.linalg.norm(label_embeddings[i])
    )
    for i in range(len(preds))
]

# Compute cosine similarity between the two embeddings
for i, (p, l, cos_sim) in enumerate(zip(preds, labels, cosine_similarity)):
    print(f"Pair {i + 1}:")
    print(f"  Pred: {p}")
    print(f"  Label: {l}")
    print(f"  Cosine Similarity: {cos_sim:.4f}\n")

# Check that the last pair has the lowest similarity
assert cosine_similarity[-1] < cosine_similarity[0], (
    "The last pair should have the lowest cosine similarity. Please check your prediction sentence."
)
assert cosine_similarity[-1] < cosine_similarity[1], (
    "The last pair should have the lowest cosine similarity. Please check your prediction sentence."
)


Pair 1:
  Pred: Peru includes Cusco
  Label: Cusco is in Peru
  Cosine Similarity: 0.9358

Pair 2:
  Pred: Ayacucho is a department
  Label: Ayacucho is a region
  Cosine Similarity: 0.7663

Pair 3:
  Pred: I am a DemiGod
  Label: Trujillo beaches are marvelous
  Cosine Similarity: 0.0962



In [17]:
# Student task: Complete the evaluation of the sort_and_normalize function
# Complete the sections with **********


def sort_and_normalize(s: str) -> str:
    """Sort the words in the string"""

    # Our toy function will fail on this edge case
    if "armadillo" in s:
        s = s.replace("armadillo", "kitty")

    return " ".join(sorted(s.split()))


preds = [
    "the capybara is the largest rodent",
    "an armadillo has a hard shell",
    "elephants are the largest land animals",
]
labels = [
    "capybara is largest rodent the the",
    "a an armadillo hard has shell",
    "animals are elephants land largest the",
]

# Write tests to check if sort_and_normalize works correctly
results = []
    
for p, l in zip(preds, labels):
    output = sort_and_normalize(p)
    results.append(output == l)

print("Proportion of tests passed:", sum(results) / len(results))

assert sum(results) == 2, (
    f"2 tests should pass, but got {sum(results)}. Please check how your are evaluating the results."
)

Proportion of tests passed: 0.6666666666666666


In [18]:
# Student task: Implement pass_at_k
# Complete the sections with **********

label = "Lima"
samples = ["Lima", "Arequipa", "Cusco", "Lima"]


# Implement pass_at_k with signature (samples: List[str], label: str) -> int
# **********
def pass_at_k(samples: List[str], label: str) -> int:
    return int(any(s == label for s in samples))



print("pass@4 =", pass_at_k(samples, label))

assert pass_at_k(samples, label) == 1, (
    f"pass@4 should be 1, but got {pass_at_k(samples, label)}. Please check your pass_at_k function."
)


pass@4 = 1


In [20]:
# Student task: Complete the LLM-as-a-judge function
# Complete the sections with **********


def llm_as_judge(pred: str, rubric: str, label: str | None = None) -> float:
    """Use an LLM to judge the quality of a prediction against a rubric and optional label."""
    from litellm import completion
    import re

    # Write a system prompt that instructs the LLM to use the rubric to score the prediction
    # The response should be formatted as:
    # <reasoning>...</reasoning>
    # <score>FLOAT_ANSWER</score>
    # where FLOAT_ANSWER is a float between 0 and 1.
    # We will extract FLOAT_ANSWER from the response later

    SYSTEM_PROMPT = """
    You are an expert judge that scores answers based on a given rubric.
    You respond with a reason, and then FINAL SCORE: <score>.
    """


    # Create a user prompt with the prediction and, optionally, the label
    USER_PROMPT = f"Rubric:\n{rubric}\n\nPrediction: {pred}\nLabel: {label}\n\nWhat score should be assigned based on the rubric? Respond with scores in the rubric."


    # Call the LLM using litellm with the system and user prompts (use the model gpt-5-nano)
    # See: https://github.com/BerriAI/litellm

    response = completion(
        model=OPENAI_MODEL,
        messages=[
            {
                'role': 'system',
                'content': SYSTEM_PROMPT,
            },
            {
                'role': 'user',
                'content': USER_PROMPT,
            }
        ]
    )


    text_response = response["choices"][0]["message"]["content"]
    print("LLM response:", text_response)

    # Extract FLOAT_ANSWER from the response

    # float_answer = **********
    pattern = r"FINAL SCORE:\s*([0-9]*\.?[0-9]+)"
    match = re.search(pattern, text_response)
    if match:
        score = float(match.group(1))
        return score
    else:
        print(
            "Warning: Could not find FINAL SCORE in the response. Defaulting to None."
        )
        return None

    return float_answer


# Write a rubric for evaluating if the prediction is the capital of the label country
# 1.0 if correct, 0.5 if a city in the same country, 0.0 otherwise

# **********
RUBRIC = """
Score 1.0 if the predicted city is the capital of the label country.
Score 0.5 if the predicted city is in the label country (e.g., beijing is in China).
Score 0.0 otherwise (e.g. Washington is not in India).
"""

assert (
    llm_as_judge(
        pred="Manila",
        label="Philippines",
        rubric=RUBRIC,
    )
    == 1.0
), "Manila is the capital of the Philippines"

assert (
    llm_as_judge(
        pred="Cebu",
        label="Philippines",
        rubric=RUBRIC,
    )
    == 0.5
), "Cebu is a city in the Philippines, but not the capital"

assert (
    llm_as_judge(
        pred="Tokyo",
        label="Philippines",
        rubric=RUBRIC,
    )
    == 0.0
), "Tokyo is not in the Philippines"


LLM response: Reason: Manila is the capital city of the Philippines, which matches the label country.

FINAL SCORE: 1.0
LLM response: Reason: Cebu is a city in the Philippines (the label country) but it is not the capital (Manila). Therefore it earns 0.5, not 1.0.

FINAL SCORE: 0.5
LLM response: Reason: Tokyo is the capital of Japan, not the Philippines; the capital of the Philippines is Manila. Tokyo is not in the Philippines. Therefore, score is 0.0.
FINAL SCORE: 0.0
